# Procesamiento_EDA_muestra

El objetivo de este código es procesar y visualizar la muestra de audios de DAIC-WOZ de acuerdo a lo descrito en el trabajo asociado, con la finalidad de poder extraer posteriormente features acústicas y explotarse en modelos predictivos

## 0. Setup
---

In [1]:
import os
import glob

import pandas as pd
import numpy as np
import librosa

In [2]:
# Path con dataset
path_sample = r'D:/DAIZ-WOZ/base/'

# Archivos
path_labels_train = os.path.join(path_sample, 'Docs/train_split_Depression_AVEC2017.csv')
path_labels_dev = os.path.join(path_sample, 'Docs/dev_split_Depression_AVEC2017.csv')
path_labels_test = os.path.join(path_sample, 'Docs/full_test_split.csv')

# Speakers entrevista
name_paciente = 'Participant'
name_entrevistador = 'Ellie'

## 1. Muestra en crudo
---

### 1.1 Carga

In [3]:
# Carga dfs train/dev
df_train = pd.read_csv(path_labels_train)
df_train['split'] = 'train'

df_dev   = pd.read_csv(path_labels_dev)
df_dev['split'] = 'dev'

# Concat de ambos
df_full = pd.concat([df_train, df_dev], ignore_index=True)
df_full

,Participant_ID,PHQ8_Binary,PHQ8_Score,Gender,PHQ8_NoInterest,PHQ8_Depressed,PHQ8_Sleep,PHQ8_Tired,PHQ8_Appetite,PHQ8_Failure,PHQ8_Concentrating,PHQ8_Moving,split
0,303,0,0,0,0,0,0.0,0,0,0,0,0,train
1,304,0,6,0,0,1,1.0,2,2,0,0,0,train
2,305,0,7,1,0,1,1.0,2,2,1,0,0,train
3,310,0,4,1,1,1,0.0,0,0,1,1,0,train
4,312,0,2,1,0,0,1.0,1,0,0,0,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,483,1,15,1,0,1,3.0,3,3,2,3,0,dev
138,484,0,9,0,1,1,0.0,3,1,2,1,0,dev
139,489,0,3,1,0,1,2.0,0,0,0,0,0,dev
140,490,0,2,1,0,1,0.0,1,0,0,0,0,dev


In [4]:
# Se filtran columnas de interés y se añade split de test
df_full = df_full[['Participant_ID', 'PHQ8_Binary', 'PHQ8_Score', 'Gender', 'split']]

df_test = pd.read_csv(path_labels_test)
df_test['split'] = 'test'
df_test = df_test.rename(
    columns={
        'PHQ_Binary': 'PHQ8_Binary',
        'PHQ_Score': 'PHQ8_Score'
    }
)

df_full = pd.concat([df_full, df_test], ignore_index=True)

# Formateo columnas
df_full.columns = df_full.columns.str.strip().str.lower()
df_full = df_full.sort_values(by='participant_id').reset_index(drop=True)
df_full

,participant_id,phq8_binary,phq8_score,gender,split
0,300,0,2,1,test
1,301,0,3,1,test
2,302,0,4,1,dev
3,303,0,0,0,train
4,304,0,6,0,train
...,...,...,...,...,...
184,488,0,0,0,train
185,489,0,3,1,dev
186,490,0,2,1,dev
187,491,0,8,0,train


### 1.2 Cálculo de variables

In [5]:
# Path con el audio
df_full['path_audio'] = [path_sample + str(N) + '_P/' + str(N) + '_AUDIO.wav' for N in df_full['participant_id'].values]

# Path con la transcripción
df_full['path_transcript'] = [path_sample + str(N) + '_P/' + str(N) + '_TRANSCRIPT.csv' for N in df_full['participant_id'].values]
df_full

,participant_id,phq8_binary,phq8_score,gender,split,path_audio,path_transcript
0,300,0,2,1,test,D:/DAIZ-WOZ/base/300_P/300_AUDIO.wav,D:/DAIZ-WOZ/base/300_P/300_TRANSCRIPT.csv
1,301,0,3,1,test,D:/DAIZ-WOZ/base/301_P/301_AUDIO.wav,D:/DAIZ-WOZ/base/301_P/301_TRANSCRIPT.csv
2,302,0,4,1,dev,D:/DAIZ-WOZ/base/302_P/302_AUDIO.wav,D:/DAIZ-WOZ/base/302_P/302_TRANSCRIPT.csv
3,303,0,0,0,train,D:/DAIZ-WOZ/base/303_P/303_AUDIO.wav,D:/DAIZ-WOZ/base/303_P/303_TRANSCRIPT.csv
4,304,0,6,0,train,D:/DAIZ-WOZ/base/304_P/304_AUDIO.wav,D:/DAIZ-WOZ/base/304_P/304_TRANSCRIPT.csv
...,...,...,...,...,...,...,...
184,488,0,0,0,train,D:/DAIZ-WOZ/base/488_P/488_AUDIO.wav,D:/DAIZ-WOZ/base/488_P/488_TRANSCRIPT.csv
185,489,0,3,1,dev,D:/DAIZ-WOZ/base/489_P/489_AUDIO.wav,D:/DAIZ-WOZ/base/489_P/489_TRANSCRIPT.csv
186,490,0,2,1,dev,D:/DAIZ-WOZ/base/490_P/490_AUDIO.wav,D:/DAIZ-WOZ/base/490_P/490_TRANSCRIPT.csv
187,491,0,8,0,train,D:/DAIZ-WOZ/base/491_P/491_AUDIO.wav,D:/DAIZ-WOZ/base/491_P/491_TRANSCRIPT.csv


Una vez cargado el dataset, se calculan variables para realizar un EDA preliminar pre procesamiento. Las variables son:

* <u>Archivo de audio</u>
* **Duración del audio**: duración del archivo de audio en minutos

* <u>Transcripciones</u>
    * <u>Tiempo hablado</u>
        * **Duración de la entrevista**: tiempo entre el comienzo de la primera intervención y el final de la última
        * **Número de turnos por participante**: número de veces en que cada participante habla
        * **Tiempo total hablado por participante**: suma de duración de los turnos
        * **Media de tiempo hablado por participante**: media de tiempo de duración de los turnos
        * **Desviación estandar de tiempo hablado por participante**: desviación estándar de tiempo de duración de los turnos
        * **Ratio de habla respecto de total**: porcentaje de habla de cada participante respecto del tiempo total (pausa + turno/hablado)
    * <u>Pausas</u>
        * **Tiempo total de pausas**: silencio entre turnos (pausas del paciente, latencia del entrevistador...)
        * **Ratio de pausa respecto del total hablado**: porcentaje de pausa respecto del tiempo total (pausa + turno/hablado)

In [6]:
def get_audio_duration_min(path_audio):
    """
    UDF: devuelve duración en minutos de audio dada su ruta
    """

    if not isinstance(path_audio, str) or not os.path.exists(path_audio):
        return np.nan
    
    try:
        return librosa.get_duration(path=path_audio) / 60
    
    except Exception:
        return np.nan


def get_transcript_stats(transcript_path):
    """
    UDF: calcula métricas de tiempo de habla y pausas a partir del 
    csv de transcript del DAIC-WOZ
    """

    # Variables a calcular
    cols = [
        # Tiempo hablado - Total
        'durSession',

        # Paciente
            # Tiempo hablado
        'nTurns_pat',
        'sumSpeech_pat',
        'avgSpeech_pat',
        'stdSpeech_pat',
        'ratioSpeech_pat',
            # Pausas
        'sumPause_pat',
        'ratioPause_pat',

        # Asistente
            # Tiempo hablado
        'nTurns_asist',
        'sumSpeech_asist',
        'avgSpeech_asist',
        'stdSpeech_asist',
        'ratioSpeech_asist',
            # Pausas
        'sumPause_asist',
        'ratioPause_asist'
    ]

    empty = pd.Series({c: np.nan for c in cols})
 
    if not isinstance(transcript_path, str) or not os.path.exists(transcript_path):
        return empty
 
    try:
        df = pd.read_csv(transcript_path, sep='\t')
        df.columns = df.columns.str.strip()
 
        df = df.sort_values('start_time').reset_index(drop=True)
        df['duration'] = (df['stop_time'] - df['start_time']) / 60
 
        # Duración total de sesión
        session_duration = (df['stop_time'].iloc[-1] - df['start_time'].iloc[0]) / 60
 
        # Métricas de habla por speaker
        def speaker_stats(speaker_label):
            turns = df[df['speaker'] == speaker_label]['duration']
            total = turns.sum()
            return [
                len(turns),
                total,
                turns.mean() if len(turns) > 0 else np.nan,
                turns.std()  if len(turns) > 1 else np.nan,
                total / session_duration if session_duration > 0 else np.nan,
            ]
 
        stats_pat   = speaker_stats(name_paciente)
        stats_asist = speaker_stats(name_entrevistador)
 
        # Métricas de pausas
        # Solo se consideran pares de speakers distintos y gaps positivos
        pause_pat   = []  # Ellie -> Paciente
        pause_asist = []  # Paciente -> Ellie
 
        for i in range(len(df) - 1):
            curr = df.iloc[i]
            nxt  = df.iloc[i + 1]
            gap  = (nxt['start_time'] - curr['stop_time']) / 60
 
            if gap <= 0:
                continue
 
            if curr['speaker'] == name_entrevistador and nxt['speaker'] == name_paciente:
                pause_pat.append(gap)
            elif curr['speaker'] == name_paciente and nxt['speaker'] == name_entrevistador:
                pause_asist.append(gap)
 
        sum_pause_pat   = sum(pause_pat)
        sum_pause_asist = sum(pause_asist)
 
        return pd.Series({
            'durSession':       session_duration,
 
            'nTurns_pat':       stats_pat[0],
            'sumSpeech_pat':    stats_pat[1],
            'avgSpeech_pat':    stats_pat[2],
            'stdSpeech_pat':    stats_pat[3],
            'ratioSpeech_pat':  stats_pat[4],

            'sumPause_pat':     sum_pause_pat,
            'ratioPause_pat':   sum_pause_pat / session_duration if session_duration > 0 else np.nan,
 
            'nTurns_asist':     stats_asist[0],
            'sumSpeech_asist':  stats_asist[1],
            'avgSpeech_asist':  stats_asist[2],
            'stdSpeech_asist':  stats_asist[3],
            'ratioSpeech_asist': stats_asist[4],

            'sumPause_asist':   sum_pause_asist,
            'ratioPause_asist': sum_pause_asist / session_duration if session_duration > 0 else np.nan,
        })
 
    except Exception as e:
        print(f'Error procesando {transcript_path}: {e}')
        return empty

In [7]:
df_full['duration_audio'] = df_full['path_audio'].apply(get_audio_duration_min)
df_full = df_full.join(df_full['path_transcript'].apply(get_transcript_stats))

df_full

c:\Users\tblxa\Desktop\Master\UNIR_IA_TFE\unir_tfm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,participant_id,phq8_binary,phq8_score,gender,split,path_audio,path_transcript,duration_audio,durSession,nTurns_pat,...,ratioSpeech_pat,sumPause_pat,ratioPause_pat,nTurns_asist,sumSpeech_asist,avgSpeech_asist,stdSpeech_asist,ratioSpeech_asist,sumPause_asist,ratioPause_asist
0,300,0,2,1,test,D:/DAIZ-WOZ/base/300_P/300_AUDIO.wav,D:/DAIZ-WOZ/base/300_P/300_TRANSCRIPT.csv,10.808333,9.744667,87.0,...,0.266402,1.561833,0.160276,87.0,2.347333,0.026981,0.015360,0.240884,1.751000,0.179688
1,301,0,3,1,test,D:/DAIZ-WOZ/base/301_P/301_AUDIO.wav,D:/DAIZ-WOZ/base/301_P/301_TRANSCRIPT.csv,13.731667,12.906667,104.0,...,0.613946,0.617833,0.047869,77.0,1.632500,0.021201,0.021970,0.126485,1.007833,0.078086
2,302,0,4,1,dev,D:/DAIZ-WOZ/base/302_P/302_AUDIO.wav,D:/DAIZ-WOZ/base/302_P/302_TRANSCRIPT.csv,12.646667,11.282833,97.0,...,0.308630,1.605283,0.142277,89.0,1.889883,0.021235,0.020190,0.167501,1.863117,0.165128
3,303,0,0,0,train,D:/DAIZ-WOZ/base/303_P/303_AUDIO.wav,D:/DAIZ-WOZ/base/303_P/303_TRANSCRIPT.csv,16.421667,15.568333,103.0,...,0.688288,0.817667,0.052521,88.0,2.470500,0.028074,0.039247,0.158688,0.617500,0.039664
4,304,0,6,0,train,D:/DAIZ-WOZ/base/304_P/304_AUDIO.wav,D:/DAIZ-WOZ/base/304_P/304_TRANSCRIPT.csv,13.210000,12.000667,104.0,...,0.503583,0.921333,0.076774,100.0,2.735000,0.027350,0.025109,0.227904,1.061833,0.088481
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184,488,0,0,0,train,D:/DAIZ-WOZ/base/488_P/488_AUDIO.wav,D:/DAIZ-WOZ/base/488_P/488_TRANSCRIPT.csv,14.748333,14.257100,138.0,...,0.493894,0.690133,0.048406,64.0,2.253783,0.035215,0.050347,0.158081,1.625583,0.114019
185,489,0,3,1,dev,D:/DAIZ-WOZ/base/489_P/489_AUDIO.wav,D:/DAIZ-WOZ/base/489_P/489_TRANSCRIPT.csv,11.745000,10.990000,117.0,...,0.256005,2.073383,0.188661,85.0,2.720983,0.032012,0.045056,0.247587,1.744150,0.158703
186,490,0,2,1,dev,D:/DAIZ-WOZ/base/490_P/490_AUDIO.wav,D:/DAIZ-WOZ/base/490_P/490_TRANSCRIPT.csv,11.521667,11.066050,97.0,...,0.279985,1.812183,0.163761,77.0,2.498617,0.032450,0.042647,0.225791,1.690817,0.152793
187,491,0,8,0,train,D:/DAIZ-WOZ/base/491_P/491_AUDIO.wav,D:/DAIZ-WOZ/base/491_P/491_TRANSCRIPT.csv,14.695000,13.519000,146.0,...,0.509875,0.518617,0.038362,85.0,2.719267,0.031991,0.044806,0.201144,1.399117,0.103493


### 1.3 Visualización

## 2. Procesamiento Muestra
---